# NOTE:

first we will create BASIC MODEL then we will try to improve it.


In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 

### Modelling 

from sklearn.metrics import mean_squared_error,r2_score 
from sklearn.neighbors import KNeighborsRegressor

from sklearn. tree import DecisionTreeRegressor
from sklearn. ensemble import RandomForestRegressor,AdaBoostRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR 
from xgboost import XGBRegressor
#from catboost import CatBoostRegressor
import warnings



In [2]:
df = pd.read_csv('data/stud.csv')

In [3]:
df

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


### Preparing X and y vairables


In [4]:
X = df.drop(columns=['math_score'],axis = 1) 

In [5]:
y = df['math_score']

In [6]:
## define numerical & categorical columns 

num_feature = [feature for feature in df.columns if df[feature].dtype != 'O']
cat_feature = [feature for feature in df.columns if df[feature].dtype == 'O']



print('we have {} numerical features :{}'.format(len(num_feature),num_feature))
print('we have {} categorical features :{}'.format(len(cat_feature),cat_feature))


we have 3 numerical features :['math_score', 'reading_score', 'writing_score']
we have 5 categorical features :['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']


In [7]:
num_feature= X.select_dtypes(exclude='object').columns
cat_feature = X.select_dtypes(include='object').columns

from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transf = StandardScaler()
Oh_transf = OneHotEncoder()

preprocessor = ColumnTransformer([
("StandardScaler",numeric_transf,num_feature),
('OneHotEncoder',Oh_transf,cat_feature)

])



In [8]:
X= preprocessor.fit_transform(X)

In [9]:
X.shape

(1000, 19)

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape

((800, 19), (200, 19))

### Create an Evaluate Function to give all metrics after the model Training

In [12]:
def evaluate(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mean_squared_error(true,predicted))
    r2_scr =r2_score(true,predicted)

    return mae,mse,rmse,r2_scr


    

    

In [24]:
models= {
"Linear Regression": LinearRegression(),
"Lasso": Lasso(),
"Ridge": Ridge(),
"K-Neighbors Regressor": KNeighborsRegressor(),
"Decision Tree": DecisionTreeRegressor(),
"Random Forest Regressor": RandomForestRegressor(),
"XGBRegressor": XGBRegressor(),
#"CatBoosting Regressor": CatBoostRegressor(verbose=False),
"AdaBoost Regressor": AdaBoostRegressor()
}


model_list=[]
r2_list = []


for i in range(len(list(models))):
    
    model = list(models.values())[i]
    model.fit(X_train,y_train)
    ## prediction :
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Evaluate train and test dataset:
    model_train_mae,model_train_mse,model_train_rmse,model_train_r2= evaluate(y_train,y_train_pred)
    model_test_mae,model_test_mse,model_test_rmse,model_test_r2= evaluate(y_test,y_test_pred)


    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("-R2.Score: {:.4f}".format (model_train_r2))

    print("---"*34)

    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("-R2.Score: {:.4f}".format(model_test_r2))

    r2_list.append(model_test_r2)

    print("=="*32)
    print("\n")










    

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3231
- Mean Absolute Error: 4.2667
-R2.Score: 0.8743
------------------------------------------------------------------------------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3940
- Mean Absolute Error: 4.2148
-R2.Score: 0.8804


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5938
- Mean Absolute Error: 5.2063
-R2.Score: 0.8071
------------------------------------------------------------------------------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5197
- Mean Absolute Error: 5.1579
-R2.Score: 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2650
-R2.Score: 0.8743
------------------------------------------------------------------------------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3904
- Mean Absolute

In [25]:
pd.DataFrame(list(zip(model_list,r2_list)),columns=['model_name','R2_score']).sort_values(by=['R2_score'])

,model_name,R2_score
4,Decision Tree,0.731012
3,K-Neighbors Regressor,0.783770
6,XGBRegressor,0.821220
1,Lasso,0.825320
7,AdaBoost Regressor,0.844860
5,Random Forest Regressor,0.851112
0,Linear Regression,0.880433
2,Ridge,0.880593
